# Fixed MLP: exact-loader five-seed reproduction

## tl;dr

This notebook applies one fixed MLP configuration to the corrected TreeLSTM(6)/TreeMoCo exact data split. It generates or loads the unchanged legacy 64-dimensional spectral vectors from raw SWC files and runs the formal protocol on three datasets with five fixed seeds.

- Equal-dataset mean test accuracy across five seeds: **65.7861% ± 0.8114%** (sample standard deviation of the five seed-level scores)
- Equal-dataset mean macro-F1 across five seeds: **61.9361% ± 0.8894%**
- Parameter count: **11,092** for ACT-4/JML-4 and **11,190** for BIL-6; the difference comes only from the number of output classes

**Evaluation note:** folds 8–9 are also used for checkpoint selection, so these values are test-selected reproduction results rather than estimates from an independent blind test set.

## Context and method

The fixed protocol uses folds 0–7 for training and folds 8–9 for testing; seeds `42, 1453, 666, 114514, 1919810`; at most 800 epochs; and patience 250. Each run selects the epoch with the highest test accuracy, breaking ties by macro-F1 and then lower test loss.

The model is a compact `64 → 96 → 48 → C` LeakyReLU MLP with one documented configuration. It does not use class weights or an alternative loss.

In [1]:
from __future__ import annotations

import copy
import glob
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.linalg import eigh, pinvh
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents, Path(r"E:\research\connectome")]:
        if (candidate / "external" / "treemoco" / "data" / "info").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate the connectome repository")


REPO_ROOT = find_repo_root()
TREEMOCO_ROOT = REPO_ROOT / "external" / "treemoco"
DATA_ROOT = TREEMOCO_ROOT / "data"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "mlp_champion_reproduction" / "outputs"
CACHE_ROOT = REPO_ROOT / "notebooks" / "V.alpha_MLP_TreeLSTM6_exact_loader_outputs" / "feature_cache"
RUN_ROOT = OUTPUT_ROOT / "runs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FORMAL_SEEDS = (42, 1453, 666, 114514, 1919810)
MAX_EPOCHS = 800
PATIENCE = 250
CHAMPION = {
    "hidden_dims": (96, 48),
    "activation": "leaky_relu",
    "dropout": 0.05,
    "batch_size": 32,
    "label_smoothing": 0.05,
    "lr": 0.003,
    "weight_decay": 0.000001,
}
REFERENCE_EQUAL_MEAN = 0.6578613890097144
SELECTION_RULE = "maximum test accuracy per epoch; ties: macro-F1, then lower test loss"

DATASET_CONFIG = {
    "ACT-4": {
        "raw_dir": DATA_ROOT / "raw" / "allen_cell_type" / "swc",
        "csv": DATA_ROOT / "info" / "ACT_info_swc_10folds.csv",
        "classes": ("Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6"),
    },
    "JML-4": {
        "raw_dir": DATA_ROOT / "raw" / "janelia_mouselight" / "swc",
        "csv": DATA_ROOT / "info" / "JML_info_swc_10folds.csv",
        "classes": ("Isocortex_layer23", "Isocortex_layer5", "Isocortex_layer6", "VPM"),
    },
    "BIL-6": {
        "raw_dir": DATA_ROOT / "raw" / "bil" / "swc",
        "csv": DATA_ROOT / "info" / "BIL_info_swc_10folds.csv",
        "classes": ("CP", "Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6", "VPM"),
    },
}

print(f"Repository: {REPO_ROOT}")
print(f"Device: {DEVICE}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Champion: {CHAMPION}")


Repository: E:\research\connectome
Device: cuda
Output: E:\research\connectome\notebooks\mlp_champion_reproduction\outputs
Champion: {'hidden_dims': (96, 48), 'activation': 'leaky_relu', 'dropout': 0.05, 'batch_size': 32, 'label_smoothing': 0.05, 'lr': 0.003, 'weight_decay': 1e-06}


C:\Users\david\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

### 1. Rebuild the exact TreeLSTM(6) sample manifest

Samples come from the official 10-fold CSV files. A row is included only when `swc__fname` exactly matches a filename in the corresponding raw directory. Path overlap and content-hash overlap between the training and test sets must both be zero.

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_label(value) -> str:
    return str(value).replace("2/3", "23")


def exact_split_manifest(dataset_name: str, phase: str) -> pd.DataFrame:
    """TreeLSTM(6) loader: CSV fold plus exact raw filename only."""
    if phase not in {"train", "test"}:
        raise ValueError("phase must be train or test")
    config = DATASET_CONFIG[dataset_name]
    classes = config["classes"]
    label_map = {name: index for index, name in enumerate(classes)}
    frame = pd.read_csv(config["csv"])
    local_files = set(glob.glob(os.path.join(str(config["raw_dir"]), "*.swc")))

    rows = []
    for source_index, record in frame.iterrows():
        label = canonical_label(record["structure_merge__acronym"])
        if label not in label_map:
            continue
        fold = int(float(record["model__fold"]))
        if phase == "train" and fold >= 8:
            continue
        if phase == "test" and fold < 8:
            continue

        filename = str(record["swc__fname"])
        exact_path_text = os.path.join(str(config["raw_dir"]), filename)
        if exact_path_text not in local_files:
            continue
        path = Path(exact_path_text).resolve()
        rows.append(
            {
                "dataset": dataset_name,
                "phase": phase,
                "source_index": int(source_index),
                "fold": fold,
                "label_id": label_map[label],
                "label_name": label,
                "filename": filename,
                "file_path": str(path),
                "is_reg": filename.lower().endswith("_reg.swc"),
                "content_hash": sha256_file(path),
            }
        )

    manifest = pd.DataFrame(rows)
    if manifest.empty:
        raise RuntimeError(f"No exact files selected for {dataset_name} {phase}")
    allowed = set(range(8)) if phase == "train" else {8, 9}
    if not set(manifest["fold"]).issubset(allowed):
        raise AssertionError("Forbidden fold entered a split")
    if manifest["file_path"].duplicated().any():
        raise AssertionError("Duplicate exact file inside a split")
    if set(manifest["label_id"]) != set(range(len(classes))):
        raise AssertionError("A configured class is absent")
    return manifest


def audit_exact_loader(dataset_name: str, train: pd.DataFrame, test: pd.DataFrame) -> dict:
    path_overlap = set(train["file_path"]) & set(test["file_path"])
    hash_overlap = set(train["content_hash"]) & set(test["content_hash"])
    if path_overlap or hash_overlap:
        raise AssertionError("Train/test leakage detected")
    if dataset_name == "BIL-6" and not pd.concat([train, test])["is_reg"].all():
        raise AssertionError("BIL contains a non-registered SWC")
    return {
        "dataset": dataset_name,
        "loader": "TreeLSTM(6) CSV folds + exact raw filename",
        "train_count": len(train),
        "test_count": len(test),
        "train_folds": ",".join(map(str, sorted(train["fold"].unique()))),
        "test_folds": ",".join(map(str, sorted(test["fold"].unique()))),
        "train_test_path_overlap": len(path_overlap),
        "train_test_content_overlap": len(hash_overlap),
        "reg_fraction": float(pd.concat([train, test])["is_reg"].mean()),
    }


### 2. Generate or load the original 64-dimensional spectrum

Spectral feature generation is unchanged: build the SWC skeleton graph, use `epsilon=20`, `tau=5`, and bridge-noise affinity `1e9`, then retain the first 64 absolute eigenvalues of the Green-distance matrix. The content-addressed cache only avoids recomputing identical raw inputs.

In [3]:
class RawExactSpectralExtractor:
    """The old 64D spectrum applied to the exact raw SWC selected by TreeLSTM(6)."""

    def __init__(self, path: Path):
        self.graph = nx.Graph()
        self.coords = {}
        self.root = None
        pending_edges = []
        with path.open("r", encoding="utf-8", errors="ignore") as handle:
            for line in handle:
                if not line.strip() or line.startswith("#"):
                    continue
                parts = line.split()
                if len(parts) < 7:
                    continue
                node = int(parts[0])
                xyz = np.array([float(parts[2]), float(parts[3]), float(parts[4])], dtype=np.float64)
                parent = int(parts[6])
                self.coords[node] = xyz
                self.graph.add_node(node)
                if parent == -1:
                    self.root = node
                else:
                    pending_edges.append((node, parent))
        if self.root is None and self.graph.number_of_nodes():
            self.root = next(iter(self.graph.nodes()))
        for node, parent in pending_edges:
            if node in self.coords and parent in self.coords:
                length = max(float(np.linalg.norm(self.coords[node] - self.coords[parent])), 1e-5)
                self.graph.add_edge(node, parent, weight=length)
        if not self.graph.number_of_nodes():
            raise ValueError(f"Empty SWC: {path}")
        largest = max(nx.connected_components(self.graph), key=len)
        self.graph = self.graph.subgraph(largest).copy()
        if self.root not in self.graph:
            self.root = next(iter(self.graph.nodes()))
        self._skeletonize()

    def _skeletonize(self):
        degree_two = [node for node in self.graph if self.graph.degree(node) == 2 and node != self.root]
        for node in degree_two:
            if node not in self.graph or self.graph.degree(node) != 2:
                continue
            left, right = list(self.graph.neighbors(node))
            length = self.graph[left][node]["weight"] + self.graph[node][right]["weight"]
            if self.graph.has_edge(left, right):
                self.graph[left][right]["weight"] = min(self.graph[left][right]["weight"], length)
            else:
                self.graph.add_edge(left, right, weight=length)
            self.graph.remove_node(node)

    def signature(self, epsilon=20.0, tau=5.0, top_k=64):
        graph = self.graph.copy()
        leaves = [node for node, degree in graph.degree() if degree == 1 and node != self.root]
        for leaf in leaves:
            distance = float(np.linalg.norm(self.coords[leaf] - self.coords[self.root]))
            if distance < epsilon and not graph.has_edge(leaf, self.root):
                graph.add_edge(leaf, self.root, weight=max(distance, 1e-5))

        bridges = {frozenset(edge) for edge in nx.bridges(graph)}
        nodes = list(graph.nodes())
        if len(nodes) < 2:
            return np.zeros(top_k, dtype=np.float64)
        index = {node: i for i, node in enumerate(nodes)}
        affinity = np.zeros((len(nodes), len(nodes)), dtype=np.float64)
        for left, right, data in graph.edges(data=True):
            i, j = index[left], index[right]
            length = max(float(data["weight"]), 1e-5)
            is_noise = frozenset((left, right)) in bridges and length < tau
            affinity[i, j] = affinity[j, i] = 1e9 if is_noise else 1.0 / length
        laplacian = np.diag(affinity.sum(axis=1)) - affinity
        laplacian_pinv = pinvh(laplacian)
        diagonal = np.diag(laplacian_pinv)
        green_distance = diagonal[:, None] + diagonal[None, :] - 2.0 * laplacian_pinv
        eigenvalues = np.sort(np.abs(eigh(green_distance, eigvals_only=True)))[::-1]
        result = np.zeros(top_k, dtype=np.float64)
        result[: min(top_k, len(eigenvalues))] = eigenvalues[:top_k]
        return result


def manifest_fingerprint(manifest: pd.DataFrame) -> str:
    payload = {
        "extractor": "TreeLSTM6_exact_raw_old64_eps20_tau5_affinity1e9_v1",
        "samples": manifest[["file_path", "content_hash", "label_id", "fold"]].to_dict("records"),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:20]


def build_or_load_features(dataset_name: str, phase: str, manifest: pd.DataFrame):
    fingerprint = manifest_fingerprint(manifest)
    cache_path = CACHE_ROOT / f"{dataset_name}_{phase}_{fingerprint}.npz"
    if cache_path.is_file():
        with np.load(cache_path, allow_pickle=False) as cached:
            matrix = np.asarray(cached["X"], dtype=np.float64)
        if matrix.shape != (len(manifest), 64):
            raise ValueError("Malformed feature cache")
        return matrix, True, cache_path

    rows = []
    for path_text in tqdm(manifest["file_path"], desc=f"Old64 {dataset_name} {phase}"):
        signature = RawExactSpectralExtractor(Path(path_text)).signature()
        if not np.isfinite(signature).all() or float(signature.sum()) == 0.0:
            raise ValueError(f"Invalid spectrum: {path_text}")
        rows.append(signature)
    matrix = np.asarray(rows, dtype=np.float64)
    np.savez_compressed(cache_path, X=matrix)
    return matrix, False, cache_path


bundles = {}
audit_rows = []
for dataset_name in DATASET_CONFIG:
    train_manifest = exact_split_manifest(dataset_name, "train")
    test_manifest = exact_split_manifest(dataset_name, "test")
    audit = audit_exact_loader(dataset_name, train_manifest, test_manifest)
    x_train, train_cache_hit, train_cache = build_or_load_features(dataset_name, "train", train_manifest)
    x_test, test_cache_hit, test_cache = build_or_load_features(dataset_name, "test", test_manifest)
    audit.update(
        {
            "feature_dim": x_train.shape[1],
            "train_cache_hit": train_cache_hit,
            "test_cache_hit": test_cache_hit,
            "train_cache": str(train_cache),
            "test_cache": str(test_cache),
        }
    )
    train_manifest.to_csv(OUTPUT_ROOT / f"{dataset_name}_train_manifest.csv", index=False)
    test_manifest.to_csv(OUTPUT_ROOT / f"{dataset_name}_test_manifest.csv", index=False)
    bundles[dataset_name] = {
        "train_manifest": train_manifest,
        "test_manifest": test_manifest,
        "x_train": x_train,
        "x_test": x_test,
        "y_train": train_manifest["label_id"].to_numpy(np.int64),
        "y_test": test_manifest["label_id"].to_numpy(np.int64),
        "classes": DATASET_CONFIG[dataset_name]["classes"],
    }
    audit_rows.append(audit)

audit_frame = pd.DataFrame(audit_rows)
audit_frame.to_csv(OUTPUT_ROOT / "dataset_audit.csv", index=False)
display(audit_frame)


,dataset,loader,train_count,test_count,train_folds,test_folds,train_test_path_overlap,train_test_content_overlap,reg_fraction,feature_dim,train_cache_hit,test_cache_hit,train_cache,test_cache
0,ACT-4,TreeLSTM(6) CSV folds + exact raw filename,400,95,"0,1,2,3,4,5,6,7","8,9",0,0,0.0,64,True,True,E:\research\connectome\notebooks\V.alpha_MLP_T...,E:\research\connectome\notebooks\V.alpha_MLP_T...
1,JML-4,TreeLSTM(6) CSV folds + exact raw filename,332,76,"0,1,2,3,4,5,6,7","8,9",0,0,0.0,64,True,True,E:\research\connectome\notebooks\V.alpha_MLP_T...,E:\research\connectome\notebooks\V.alpha_MLP_T...
2,BIL-6,TreeLSTM(6) CSV folds + exact raw filename,958,242,"0,1,2,3,4,5,6,7","8,9",0,0,1.0,64,True,True,E:\research\connectome\notebooks\V.alpha_MLP_T...,E:\research\connectome\notebooks\V.alpha_MLP_T...


## Results

### 3. Define the fixed model and training rule, then report parameter counts

In [4]:
def activation_layer(name: str):
    if name == "leaky_relu":
        return nn.LeakyReLU(0.1)
    raise ValueError(name)


class ChampionMLP(nn.Module):
    """Old64 -> 96 -> 48 -> class logits; 11,190 parameters for BIL-6."""

    def __init__(self, num_classes: int):
        super().__init__()
        layers = []
        width = 64
        for hidden in CHAMPION["hidden_dims"]:
            layers.extend(
                [
                    nn.Linear(width, hidden),
                    activation_layer(CHAMPION["activation"]),
                    nn.Dropout(CHAMPION["dropout"]),
                ]
            )
            width = hidden
        layers.append(nn.Linear(width, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, features):
        return self.net(features)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def make_loader(features, targets, shuffle: bool, seed: int):
    dataset = TensorDataset(
        torch.tensor(np.asarray(features), dtype=torch.float32),
        torch.tensor(np.asarray(targets), dtype=torch.long),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=CHAMPION["batch_size"],
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, targets, predictions = 0.0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            logits = model(features)
            total_loss += criterion(logits, labels).item() * labels.numel()
            targets.append(labels.cpu().numpy())
            predictions.append(logits.argmax(dim=1).cpu().numpy())
    y_true = np.concatenate(targets)
    y_pred = np.concatenate(predictions)
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def is_better(current, best):
    if best is None or current["accuracy"] != best["accuracy"]:
        return best is None or current["accuracy"] > best["accuracy"]
    if current["macro_f1"] != best["macro_f1"]:
        return current["macro_f1"] > best["macro_f1"]
    return current["loss"] < best["loss"]


def train_one(dataset_name: str, seed: int, bundle: dict) -> dict:
    set_seed(seed)
    feature_mean = bundle["x_train"].mean(axis=0, keepdims=True)
    feature_std = bundle["x_train"].std(axis=0, keepdims=True)
    feature_std[feature_std < 1e-8] = 1.0
    x_train = (bundle["x_train"] - feature_mean) / feature_std
    x_test = (bundle["x_test"] - feature_mean) / feature_std
    train_loader = make_loader(x_train, bundle["y_train"], True, seed)
    test_loader = make_loader(x_test, bundle["y_test"], False, seed)

    model = ChampionMLP(len(bundle["classes"])).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=CHAMPION["label_smoothing"])
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CHAMPION["lr"],
        weight_decay=CHAMPION["weight_decay"],
    )
    best, best_state, history, stale_epochs = None, None, [], 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        current = evaluate(model, test_loader, criterion)
        current["epoch"] = epoch
        history.append(
            {
                "epoch": epoch,
                "test_accuracy": current["accuracy"],
                "test_macro_f1": current["macro_f1"],
                "test_loss": current["loss"],
            }
        )
        if is_better(current, best):
            best = copy.deepcopy(current)
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break

    prefix = f"{dataset_name}_seed{seed}"
    history_frame = pd.DataFrame(history)
    history_frame["selected_checkpoint"] = history_frame["epoch"].eq(best["epoch"])
    history_frame.to_csv(RUN_ROOT / f"{prefix}_history.csv", index=False)
    torch.save(
        {
            "state_dict": best_state,
            "feature_mean": feature_mean,
            "feature_std": feature_std,
            "classes": bundle["classes"],
            "seed": seed,
            "champion": CHAMPION,
            "best": best,
            "epochs_ran": len(history),
            "patience": PATIENCE,
        },
        RUN_ROOT / f"{prefix}_best.pt",
    )
    return {
        "dataset": dataset_name,
        "seed": seed,
        "train_count": len(bundle["y_train"]),
        "test_count": len(bundle["y_test"]),
        "epochs_ran": len(history),
        "stopped_early": len(history) < MAX_EPOCHS,
        "best_epoch": best["epoch"],
        "best_test_accuracy": best["accuracy"],
        "best_test_macro_f1": best["macro_f1"],
        "best_test_loss": best["loss"],
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
    }


# Parameter count is reported explicitly because the output head depends on C.
parameter_rows = []
for dataset_name, bundle in bundles.items():
    num_classes = len(bundle["classes"])
    model_for_count = ChampionMLP(num_classes)
    layer_64_to_96 = 64 * 96 + 96
    layer_96_to_48 = 96 * 48 + 48
    output_head = 48 * num_classes + num_classes
    total_parameters = sum(parameter.numel() for parameter in model_for_count.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model_for_count.parameters() if parameter.requires_grad
    )
    assert total_parameters == layer_64_to_96 + layer_96_to_48 + output_head
    assert total_parameters == trainable_parameters
    parameter_rows.append(
        {
            "dataset": dataset_name,
            "architecture": f"64 -> 96 -> 48 -> {num_classes}",
            "input_to_hidden_parameters": layer_64_to_96,
            "hidden_to_hidden_parameters": layer_96_to_48,
            "classification_head_parameters": output_head,
            "total_parameters": total_parameters,
            "trainable_parameters": trainable_parameters,
        }
    )

parameter_frame = pd.DataFrame(parameter_rows)
parameter_frame.to_csv(OUTPUT_ROOT / "parameter_count.csv", index=False)
display(parameter_frame)


,dataset,architecture,input_to_hidden_parameters,hidden_to_hidden_parameters,classification_head_parameters,total_parameters,trainable_parameters
0,ACT-4,64 -> 96 -> 48 -> 4,6240,4656,196,11092,11092
1,JML-4,64 -> 96 -> 48 -> 4,6240,4656,196,11092,11092
2,BIL-6,64 -> 96 -> 48 -> 6,6240,4656,294,11190,11190


### 4. Run three datasets × five seeds and compute the equal-dataset mean

In [5]:
result_rows = []
for dataset_name, bundle in bundles.items():
    for seed in FORMAL_SEEDS:
        result = train_one(dataset_name, seed, bundle)
        result_rows.append(result)
        print(
            f"{dataset_name} seed={seed}: "
            f"best={result['best_test_accuracy']:.4f} "
            f"epoch={result['best_epoch']}/{result['epochs_ran']}"
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_frame = pd.DataFrame(result_rows)
results_frame.to_csv(OUTPUT_ROOT / "formal_results.csv", index=False)
summary_frame = results_frame.groupby("dataset", as_index=False).agg(
    seed_count=("seed", "size"),
    mean_test_accuracy=("best_test_accuracy", "mean"),
    std_test_accuracy=("best_test_accuracy", "std"),
    max_test_accuracy=("best_test_accuracy", "max"),
    mean_test_macro_f1=("best_test_macro_f1", "mean"),
    std_test_macro_f1=("best_test_macro_f1", "std"),
    mean_epochs_ran=("epochs_ran", "mean"),
    parameters=("parameters", "first"),
)
summary_frame.to_csv(OUTPUT_ROOT / "formal_summary.csv", index=False)

# One comparable overall score per seed: equal mean across ACT, JML, and BIL.
# The reported ± is the sample SD across these five seed-level scores.
seed_summary_frame = results_frame.groupby("seed", as_index=False).agg(
    seed_equal_dataset_accuracy=("best_test_accuracy", "mean"),
    seed_equal_dataset_macro_f1=("best_test_macro_f1", "mean"),
)
seed_summary_frame.to_csv(OUTPUT_ROOT / "seed_summary.csv", index=False)

equal_dataset_mean_accuracy = float(seed_summary_frame["seed_equal_dataset_accuracy"].mean())
equal_dataset_std_accuracy = float(seed_summary_frame["seed_equal_dataset_accuracy"].std(ddof=1))
equal_dataset_mean_macro_f1 = float(seed_summary_frame["seed_equal_dataset_macro_f1"].mean())
equal_dataset_std_macro_f1 = float(seed_summary_frame["seed_equal_dataset_macro_f1"].std(ddof=1))
reproduction_delta_pp = (equal_dataset_mean_accuracy - REFERENCE_EQUAL_MEAN) * 100
overall_frame = pd.DataFrame(
    [
        {
            "equal_dataset_mean_accuracy": equal_dataset_mean_accuracy,
            "equal_dataset_std_accuracy_across_seeds": equal_dataset_std_accuracy,
            "equal_dataset_mean_macro_f1": equal_dataset_mean_macro_f1,
            "equal_dataset_std_macro_f1_across_seeds": equal_dataset_std_macro_f1,
            "reference_equal_dataset_mean_accuracy": REFERENCE_EQUAL_MEAN,
            "delta_percentage_points": reproduction_delta_pp,
            "formal_runs": len(results_frame),
        }
    ]
)
overall_frame.to_csv(OUTPUT_ROOT / "overall_summary.csv", index=False)

protocol = {
    "loader": "TreeLSTM(6): CSV model__fold plus exact raw swc__fname",
    "split": {"train_folds": list(range(8)), "test_folds": [8, 9]},
    "seeds": list(FORMAL_SEEDS),
    "feature": "old64 spectrum: epsilon=20, tau=5, bridge-noise affinity=1e9",
    "model": CHAMPION,
    "parameter_count": {
        row["dataset"]: int(row["total_parameters"])
        for row in parameter_rows
    },
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "selection_rule": SELECTION_RULE,
    "uncertainty": "sample SD across five seed-level equal-dataset scores",
    "caveat": "folds 8-9 select checkpoints, so this is a test-selected reproduction",
}
(OUTPUT_ROOT / "protocol.json").write_text(json.dumps(protocol, indent=2, default=list), encoding="utf-8")

display(results_frame)
display(summary_frame)
display(seed_summary_frame)
display(overall_frame)


ACT-4 seed=42: best=0.6421 epoch=114/364


ACT-4 seed=1453: best=0.6316 epoch=247/497


ACT-4 seed=666: best=0.6211 epoch=155/405


ACT-4 seed=114514: best=0.6421 epoch=256/506


ACT-4 seed=1919810: best=0.6526 epoch=279/529


...

BIL-6 seed=1919810: best=0.7397 epoch=357/607


,dataset,seed,train_count,test_count,epochs_ran,stopped_early,best_epoch,best_test_accuracy,best_test_macro_f1,best_test_loss,parameters
0,ACT-4,42,400,95,364,True,114,0.642105,0.635867,1.216634,11092
1,ACT-4,1453,400,95,497,True,247,0.631579,0.627767,1.299183,11092
2,ACT-4,666,400,95,405,True,155,0.621053,0.587356,1.185677,11092
3,ACT-4,114514,400,95,506,True,256,0.642105,0.634885,1.335510,11092
4,ACT-4,1919810,400,95,529,True,279,0.652632,0.647800,1.277086,11092
5,JML-4,42,332,76,432,True,182,0.631579,0.632102,1.074617,11092
6,JML-4,1453,332,76,451,True,201,0.605263,0.600016,1.178324,11092
7,JML-4,666,332,76,371,True,121,0.592105,0.591944,1.056142,11092
8,JML-4,114514,332,76,322,True,72,0.592105,0.578046,1.042014,11092
9,JML-4,1919810,332,76,715,True,465,0.592105,0.598098,1.425779,11092


,dataset,seed_count,mean_test_accuracy,std_test_accuracy,max_test_accuracy,mean_test_macro_f1,std_test_macro_f1,mean_epochs_ran,parameters
0,ACT-4,5,0.637895,0.012002,0.652632,0.626735,0.023159,460.2,11092
1,BIL-6,5,0.733058,0.008569,0.743802,0.631307,0.024314,579.0,11190
2,JML-4,5,0.602632,0.017156,0.631579,0.600041,0.019881,458.2,11092


,seed,seed_equal_dataset_accuracy,seed_equal_dataset_macro_f1
0,42,0.666986,0.618661
1,666,0.645433,0.606197
2,1453,0.660215,0.623356
3,114514,0.655205,0.618052
4,1919810,0.661469,0.630539


,equal_dataset_mean_accuracy,equal_dataset_std_accuracy_across_seeds,equal_dataset_mean_macro_f1,equal_dataset_std_macro_f1_across_seeds,reference_equal_dataset_mean_accuracy,delta_percentage_points,formal_runs
0,0.657861,0.008114,0.619361,0.008894,0.657861,0.0,15


### 5. Check reproduction consistency

In [6]:
assert len(results_frame) == len(DATASET_CONFIG) * len(FORMAL_SEEDS) == 15
assert parameter_frame.set_index('dataset')['total_parameters'].to_dict() == {'ACT-4': 11092, 'JML-4': 11092, 'BIL-6': 11190}
assert not results_frame.duplicated(["dataset", "seed"]).any()
assert set(results_frame["seed"]) == set(FORMAL_SEEDS)
assert set(results_frame["dataset"]) == set(DATASET_CONFIG)
assert len(list(RUN_ROOT.glob("*_history.csv"))) == 15
assert len(list(RUN_ROOT.glob("*_best.pt"))) == 15

selection_mismatches = []
for row in results_frame.itertuples(index=False):
    history = pd.read_csv(RUN_ROOT / f"{row.dataset}_seed{row.seed}_history.csv")
    selected = history.sort_values(
        ["test_accuracy", "test_macro_f1", "test_loss"],
        ascending=[False, False, True],
        kind="mergesort",
    ).iloc[0]
    if int(selected["epoch"]) != int(row.best_epoch):
        selection_mismatches.append((row.dataset, row.seed))
    if bool(row.stopped_early):
        assert int(row.epochs_ran) - int(row.best_epoch) == PATIENCE

assert not selection_mismatches
assert np.isclose(
    equal_dataset_mean_accuracy, summary_frame["mean_test_accuracy"].mean(), rtol=0, atol=1e-15
)
assert len(seed_summary_frame) == len(FORMAL_SEEDS)
reference_match = bool(np.isclose(equal_dataset_mean_accuracy, REFERENCE_EQUAL_MEAN, rtol=0, atol=1e-12))
validation_frame = pd.DataFrame(
    [
        {
            "formal_runs": len(results_frame),
            "history_files": 15,
            "checkpoint_files": 15,
            "selection_rule_mismatches": len(selection_mismatches),
            "reference_match_at_1e-12": reference_match,
            "seed_level_std_accuracy": equal_dataset_std_accuracy,
            "reproduction_delta_percentage_points": reproduction_delta_pp,
        }
    ]
)
validation_frame.to_csv(OUTPUT_ROOT / "validation_summary.csv", index=False)
display(validation_frame)


,formal_runs,history_files,checkpoint_files,selection_rule_mismatches,reference_match_at_1e-12,seed_level_std_accuracy,reproduction_delta_percentage_points
0,15,15,15,0,True,0.008114,0.0


## Conclusion

- A single fixed architecture and parameter configuration is applied consistently to every run.
- The model body is `64 → 96 → 48 → C`, with 11,092 trainable parameters for ACT-4/JML-4 and 11,190 for BIL-6.
- ACT/JML/BIL are averaged with equal weight within each seed; the final `±` is the sample standard deviation of the five seed-level scores.
- Machine-readable results are stored in `outputs/formal_results.csv`, `outputs/formal_summary.csv`, `outputs/parameter_count.csv`, and `outputs/protocol.json`.